# SuSiEx with reference panel

In [1]:
import polars as pl
from pyprojroot.here import here

Prepare sumstats

In [2]:
# filter out null/unmatched variants
qc_mask = pl.any_horizontal(
    pl.all().is_null()
)

for dataset in ["mdd2024_afr", "mdd2024_eas", "mdd2024_his", "mdd2025_eur", "mdd2024_sas"]:
    sumstats = pl.scan_parquet(here(f"data/processed/sumstats/tidy/{dataset}/tidyGWAS_hivestyle"))
    susie = (sumstats
    .select(
        pl.col("CHR"),
        pl.col("RSID").alias("SNP"),
        pl.col("POS_37").alias("BP"),
        pl.col("EffectAllele").alias("A1"),
        pl.col("OtherAllele").alias("A2"),
        pl.col("B").alias("BETA"),
        pl.col("SE"),
        pl.col("P")
    )
    .filter(~qc_mask)
    )

    susie.sink_csv(here(f"data/processed/sumstats/{dataset}_hg19_susie.tsv"), separator = "\t")


Run SuSiEx

In [21]:
%%bash -s {here()}
here=$1
sumstats=${here}/data/processed/sumstats
reference=${here}/data/processed/reference

sumstats_afr=${sumstats}/mdd2024_afr_hg19_susie.tsv
sumstats_eas=${sumstats}/mdd2024_eas_hg19_susie.tsv
sumstats_his=${sumstats}/mdd2024_his_hg19_susie.tsv
sumstats_sas=${sumstats}/mdd2024_sas_hg19_susie.tsv
sumstats_eur=${sumstats}/mdd2025_eur_hg19_susie.tsv 

ref_afr=${reference}/all_hg19_AFR_chr11_61000000-63000000
ref_eas=${reference}/all_hg19_EAS_chr11_61000000-63000000
ref_amr=${reference}/all_hg19_AMR_chr11_61000000-63000000
ref_sas=${reference}/all_hg19_SAS_chr11_61000000-63000000
ref_eur=${reference}/all_hg19_EUR_chr11_61000000-63000000

ld_afr=${reference}/ld/all_hg19_AFR_chr11_61000000-63000000
ld_eas=${reference}/ld/all_hg19_EAS_chr11_61000000-63000000
ld_amr=${reference}/ld/all_hg19_AMR_chr11_61000000-63000000
ld_sas=${reference}/ld/all_hg19_SAS_chr11_61000000-63000000
ld_eur=${reference}/ld/all_hg19_EUR_chr11_61000000-63000000

mkdir -p ${reference}/ld
mkdir -p ${here}/data/results/SuSiEx

$here/vendor/SuSiEx/bin_static/SuSiEx \
  --sst_file=$sumstats_afr,$sumstats_eas,$sumstats_his,$sumstats_sas,$sumstats_eur \
  --n_gwas=70727,58319,16211,14979,1577200 \
  --ref_file=$ref_afr,$ref_eas,$ref_amr,$ref_sas,$ref_eur \
  --ld_file=$ld_afr,$ld_eas,$ld_amr,$ld_sas,$ld_eur \
  --out_dir=${here}/reports/tables/SuSiEx \
  --out_name=mdd_all_hg19_chr11_61000000-63000000 \
  --chr=11 \
  --bp=61000000,63000000 \
  --chr_col=1,1,1,1,1 \
  --snp_col=2,2,2,2,2 \
  --bp_col=3,3,3,3,3 \
  --a1_col=4,4,4,4,4 \
  --a2_col=5,5,5,5,5 \
  --eff_col=6,6,6,6,6 \
  --se_col=7,7,7,7,7 \
  --pval_col=8,8,8,8,8 \
  --plink=${here}/.pixi/envs/default/bin/plink \
  --keep-ambig=True \
  --maf=0.005 \
  --mult-step=True \
  --pval_thresh=1e-5 \
  --max_iter=500 \
  --tol=1e-4 \
  --threads=8


Software parameters:
--sst_file = /home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_afr_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_eas_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_his_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_sas_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2025_eur_hg19_susie.tsv
--ld_file = /home/madams23/Pro

jects/cvd-mh-loci/data/processed/reference/ld/all_hg19_AFR_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_EAS_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_AMR_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_SAS_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_EUR_chr11_61000000-63000000
--n_gwas = 70727,58319,16211,14979,1577200
--out_dir = /home/madams23/Projects/cvd-mh-loci/reports/tables/SuSiEx
--out_name = mdd_all_hg19_chr11_61000000-63000000
--chr = 11
--bp = 61000000,63000000
--chr_col = 1,1,1,1,1
--snp_col = 2,2,2,2,2
--bp_col = 3,3,3,3,3
--a1_col = 4,4,4,4,4
--a2_col = 5,5,5,5,5
--eff_col = 6,6,6,6,6
--se_col = 7,7,7,7,7
--pval_col = 8,8,8,8,8
--plink = /home/madams23/Projects/cvd-mh-loci/.pixi/envs/default/bin/plink
--keep-ambig = true
--mult-step = true
--precmp = true
--maf = 0.0

Parse results

In [22]:
cs = pl.read_csv(here("reports/tables/SuSiEx/mdd_all_hg19_chr11_61000000-63000000.cs"), separator="\t")
snp = pl.read_csv(here("reports/tables/SuSiEx/mdd_all_hg19_chr11_61000000-63000000.snp"), separator="\t")
summary = pl.read_csv(here("reports/tables/SuSiEx/mdd_all_hg19_chr11_61000000-63000000.summary"), separator="\t", comment_prefix="#")

In [23]:
summary

CS_ID,CS_LENGTH,CS_PURITY,MAX_PIP_SNP,BP,REF_ALLELE,ALT_ALLELE,REF_FRQ,BETA,SE,-LOG10P,MAX_PIP,POST-HOC_PROB_POP1,POST-HOC_PROB_POP2,POST-HOC_PROB_POP3,POST-HOC_PROB_POP4,POST-HOC_PROB_POP5
i64,i64,f64,str,i64,str,str,str,str,str,str,f64,f64,f64,f64,f64,i64
1,4,0.998286,"""rs28456""",61589481,"""A,A,A,A,A""","""G,G,G,G,G""","""0.8857,0.4534,0.4222,0.8727,0.…","""-0.0211841,-0.00845225,0.02271…","""0.00835653,0.00588172,0.011244…","""1.94909,0.821865,1.36261,0.544…",0.458328,0.731667,0.346873,0.432911,0.343089,1
2,32,0.811537,"""rs112830700""",61133980,"""NA,NA,G,NA,G""","""NA,NA,A,NA,A""","""NA,NA,0.93516,NA,0.8817""","""NA,NA,0.030182,NA,0.0111027""","""NA,NA,0.0225536,NA,0.00174337""","""NA,NA,0.742754,NA,9.71936""",0.334592,0.590189,0.352458,0.405294,0.326591,1


In [24]:
cs

CS_ID,SNP,BP,REF_ALLELE,ALT_ALLELE,REF_FRQ,BETA,SE,-LOG10P,CS_PIP,OVRL_PIP
i64,str,i64,str,str,str,str,str,str,f64,f64
1,"""rs28456""",61589481,"""A,A,A,A,A""","""G,G,G,G,G""","""0.8857,0.4534,0.4222,0.8727,0.…","""-0.0211841,-0.00845225,0.02271…","""0.00835653,0.00588172,0.011244…","""1.94909,0.821865,1.36261,0.544…",0.458328,0.458328
1,"""rs174548""",61571348,"""C,C,C,C,C""","""G,G,G,G,G""","""0.8216,0.4524,0.4107,0.8727,0.…","""-0.016392,-0.00798664,0.022653…","""0.00694488,0.00588284,0.011288…","""1.73849,0.757992,1.34894,0.525…",0.381283,0.381283
1,"""rs174560""",61581764,"""T,T,T,T,T""","""C,C,C,C,C""","""0.8239,0.4534,0.4121,0.8717,0.…","""-0.0159987,-0.0076735,0.022490…","""0.00698032,0.00588172,0.011283…","""1.65941,0.716661,1.33505,0.515…",0.099673,0.099673
1,"""rs174549""",61571382,"""G,G,G,G,G""","""A,A,A,A,A""","""0.97942,0.4524,0.4308,0.8747,0…","""-0.0287159,-0.00759704,0.02268…","""0.0187277,0.00588284,0.0112153…","""0.902417,0.706487,1.36543,0.53…",0.0293262,0.0293262
2,"""rs112830700""",61133980,"""NA,NA,G,NA,G""","""NA,NA,A,NA,A""","""NA,NA,0.93516,NA,0.8817""","""NA,NA,0.030182,NA,0.0111027""","""NA,NA,0.0225536,NA,0.00174337""","""NA,NA,0.742754,NA,9.71936""",0.334592,0.334592
…,…,…,…,…,…,…,…,…,…,…
2,"""rs35605696""",61231177,"""A,A,A,A,A""","""G,G,G,G,G""","""0.2652,0.0238,0.1182,0.1448,0.…","""-0.0136729,0.0181136,-0.009475…","""0.00602311,0.0192098,0.0172023…","""1.63445,0.461283,0.235249,0.07…",0.0059106,0.0059106
2,"""rs28720282""",61090956,"""NA,NA,T,NA,T""","""NA,NA,C,NA,C""","""NA,NA,0.93516,NA,0.8777""","""NA,NA,0.0270559,NA,0.010537""","""NA,NA,0.0225536,NA,0.00171852""","""NA,NA,0.637736,NA,9.05995""",0.005834,0.005834
2,"""rs3018734""",61211286,"""A,NA,A,A,A""","""G,NA,G,G,G""","""0.1433,NA,0.0793,0.0595,0.1203""","""-0.0129047,NA,-0.00425092,-0.0…","""0.00758848,NA,0.0205534,0.0244…","""1.05049,NA,0.0777168,0.0243911…",0.005606,0.005606


In [25]:
snp

BP,SNP,PIP(CS1),"LogBF(CS1,Pop1)","LogBF(CS1,Pop2)","LogBF(CS1,Pop3)","LogBF(CS1,Pop4)","LogBF(CS1,Pop5)",PIP(CS2),"LogBF(CS2,Pop1)","LogBF(CS2,Pop2)","LogBF(CS2,Pop3)","LogBF(CS2,Pop4)","LogBF(CS2,Pop5)"
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
61008633,"""rs541860247""",5.0052e-20,-0.495548,-1.3958e-10,-4.3804e-10,-2.7533e-10,-2.8249e-11,8.4877e-12,-0.626074,-1.3958e-10,-4.3804e-10,-2.7533e-10,-2.8249e-11
61013514,"""rs200615031""",3.1645e-20,-0.84742,0.884821,-0.991437,-2.7533e-10,-2.8249e-11,8.2457e-12,-0.920438,1.28798,-1.02255,-2.7533e-10,-2.8249e-11
61013552,"""rs532971636""",3.4240e-20,-0.875213,-1.3958e-10,-4.3804e-10,-2.7533e-10,-2.8249e-11,6.7752e-12,-0.851424,-1.3958e-10,-4.3804e-10,-2.7533e-10,-2.8249e-11
61014040,"""rs56148061""",1.0791e-19,-7.7104e-11,-1.3958e-10,0.272652,-2.7533e-10,-2.8249e-11,4.8959e-11,-7.7104e-11,-1.3958e-10,1.12629,-2.7533e-10,-2.8249e-11
61014431,"""rs116989755""",2.0063e-20,-7.7104e-11,-1.40973,-4.3804e-10,-2.7533e-10,-2.8249e-11,3.8345e-12,-7.7104e-11,-1.42065,-4.3804e-10,-2.7533e-10,-2.8249e-11
…,…,…,…,…,…,…,…,…,…,…,…,…,…
62999675,"""rs138738771""",1.5786e-20,-0.318016,-1.3958e-10,-1.33148,-2.7533e-10,-2.8249e-11,3.7341e-12,-0.218586,-1.3958e-10,-1.22859,-2.7533e-10,-2.8249e-11
62999828,"""rs190944738""",6.7261e-21,-1.23071,-1.3958e-10,-1.2719,-2.7533e-10,-2.8249e-11,1.2389e-12,-1.22944,-1.3958e-10,-1.32103,-2.7533e-10,-2.8249e-11
62999834,"""rs55935820""",5.9045e-20,-1.2354,-1.3958e-10,0.302678,-2.7533e-10,0.602417,1.3980e-11,-1.21348,-1.3958e-10,0.149376,-2.7533e-10,0.937062
